In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
fasta='merged.fasta'
AA_keys="ACDEFGHIKLMNPQRSTVWY" 
df_background=pd.read_csv('/home/um40/LB2_project_Group_1/Data_analysis/aa_composition_plots/background_freq.tsv',sep='\t')
df_background.sort_values(by='aa',inplace=True)
df=pd.read_csv('/home/um40/LB2_project_Group_1/Data_preparation/Cleaned_merged_dataset.tsv',sep='\t')



In [3]:
def count(fasta,data,j):
    '''counts the number of cleavage sites in the training set and returns a list of cleavage sites,
      a list of validation sequences and a list of test sequences'''
    cleavage_site_train=list()
    seq_val=list()
    seq_test=list()


    with open(fasta, 'r') as fasta:
        for line in fasta:
            if line.startswith('>'):
                UniprotID = line.rstrip()[1:]

                #take the fold 
                fold = data.loc[data['ID'] == UniprotID, 'fold'].values[0] -1 # estraggo la riga corrispondente ad UniprotID e ricavo il valore di fold per determinare a che set appartiene

                #take the class
                clas=data.loc[data['ID'] == UniprotID, 'class'].values[0]

                #take the context
                cleavage_pos = data.loc[data['ID'] == UniprotID, 'cleavage_pos'].values[0]


                start_pos = cleavage_pos - 13
                end_pos = cleavage_pos + 2

                seq = fasta.readline().strip()

                #lista per split 
                fold_train=[(i+j)%5 for i in range(2,5)]
                

                if fold in fold_train and clas==1: 
                    cleavage_site_train.append(seq[start_pos:end_pos])
                elif fold==(j+1)%5:
                    seq_val.append((seq,clas))
                elif fold==j:
                    # print('è entrato')
                    seq_test.append((seq,clas))

                    
    return cleavage_site_train,seq_val,seq_test


In [4]:
#pesudocount

def create_pswm(train_set,freq):
    '''creates a numpy array of shape (15,20)
      with the log2 of the ratio between the frequency of each amino acid
    in each position and the background frequency of that amino acid  '''
    pspm=np.ones([15,20])

    for seq in train_set:
        for i in range(len(seq)):
            pos_aa=AA_keys.find(seq[i])
            pspm[i,pos_aa]+=1

    pspm/=(len(train_set)+20)


    return np.log2(pspm/freq)

In [6]:
def score_seq(seq,pswm):
    '''scores a sequence with the pswm and returns the best score 
    and the position of the best score'''
    best_win_score,pos_best=float('-inf'),0
    final_pos=min(len(seq),90)-15
    
    #scorro sulla val_seq
    for i in range(0,final_pos):
        score=0
        
        #scorro sulla pswm
        for row in range(0,len(pswm[:,0])):


            #trovo col aa nella pswm 
            col=AA_keys.find(seq[row+i])
            
            #aggiungo allo score della window, il valore dell aa nella pswm
            score+=pswm[row,col]

        #esaurisco la window e aggiorno il miglior score solo se è il migliore 
        if best_win_score < score:
            best_win_score=score
            pos_best=i

    return best_win_score, pos_best

In [7]:
from sklearn.metrics import precision_recall_curve,confusion_matrix,ConfusionMatrixDisplay,matthews_corrcoef, f1_score,precision_score, recall_score,accuracy_score

def opt_th(y_val , y_val_scores,y_test_scores):
    '''finds the optimal threshold for classification based on F1 score  and returns the optimal threshold and the predicted labels for the test set'''
    # y: 0=NO_SP, 1=SP ; scores from PSWM scanning on validation data
    precision, recall, thresholds = precision_recall_curve(y_val , y_val_scores)
    f1 = (2 * precision * recall) / (precision + recall)

    optimal_threshold = thresholds[np.argmax(f1)]
    # classify the test set with the chosen threshold
    # y_pred = [int(s >= optimal_threshold) for s in y_test_scores]

    return optimal_threshold

In [8]:
def metrics_test(th,y_test_scores,y_test):
    y_pred = [int(s >= th) for s in y_test_scores]
    cm=confusion_matrix(y_test,y_pred)

    mcc=matthews_corrcoef(y_test,y_pred)
    f1=f1_score(y_test,y_pred)
    precision=precision_score(y_test,y_pred)
    recall=recall_score(y_test,y_pred)
    acc=accuracy_score(y_test,y_pred)

    return mcc,f1,precision,recall,acc,cm

In [9]:
freq=df_background['freq_back'].to_numpy()

df_metrics = pd.DataFrame(columns=['mcc', 'f1', 'precision', 'recall', 'acc', 'cm'])

for run in range(0,5):
    train_set,val_set,test_set=count(fasta,df,run)
    pswm=create_pswm(train_set,freq)

    y_val_scores=[]
    y_val=[]
    for (seq,clas) in val_set:
        y_val_scores.append(score_seq(seq,pswm)[0])
        y_val.append(clas)

    y_test_scores=[]
    y_test=[]
    for (seq,clas) in test_set:
        y_test_scores.append(score_seq(seq,pswm)[0])
        y_test.append(clas)


    th=opt_th(y_val , y_val_scores,y_test_scores)
    df_metrics.loc[len(df_metrics)]=metrics_test(th,y_test_scores,y_test)

In [10]:
df_metrics

,mcc,f1,precision,recall,acc,cm
0,0.720084,0.751958,0.699029,0.813559,0.938985,"[[1318, 62], [33, 144]]"
1,0.623430,0.662687,0.702532,0.627119,0.927425,"[[1333, 47], [66, 111]]"
2,0.719265,0.750716,0.757225,0.744318,0.944087,"[[1338, 42], [45, 131]]"
3,0.701424,0.730769,0.633333,0.863636,0.928021,"[[1292, 88], [24, 152]]"
4,0.697150,0.730435,0.750000,0.711864,0.940231,"[[1337, 42], [51, 126]]"


In [11]:
se_matrics=df_metrics[df_metrics.columns[:-1]].std(axis=0)/np.sqrt(len(df_metrics))
se_matrics


mcc          0.017817
f1           0.016330
precision    0.022217
recall       0.040937
acc          0.003385
dtype: float64

In [12]:
avg_metrics=df_metrics[df_metrics.columns[:-1]].mean(axis=0)
avg_metrics

mcc          0.692271
f1           0.725313
precision    0.708424
recall       0.752099
acc          0.935750
dtype: float64

In [13]:
for metrics in df_metrics.columns[:-1]:
    print(f'{metrics} = {avg_metrics[metrics]} ± {se_matrics[metrics]}')


mcc = 0.6922707932469147 ± 0.01781696451313084
f1 = 0.7253130274926821 ± 0.016329988499541783
precision = 0.7084239077285115 ± 0.022216812783105673
recall = 0.7520993836671803 ± 0.04093739590084463
acc = 0.9357498187966113 ± 0.0033846650010532393


In [ ]:
def spiderplot(df_metrics):
    categories = ['mcc', 'f1', 'precision',	'recall', 'acc']
    df_metrics = df_metrics.loc[:, categories]
    values = [1, 1, 1, 1, 1]

    # Calcolo angoli e chiusura poligono
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()

    values += values[:1]
    angles += angles[:1]

    # Plot
    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
    ax.plot(angles, values, 'o-', linewidth=2)
    ax.fill(angles, values, alpha=0.25)
    ax.set_thetagrids(np.degrees(angles[:-1]), categories)
    ax.set_ylim(0, 100)
    plt.show()

In [17]:
spiderplot(df_metrics)

UnboundLocalError: local variable 'values' referenced before assignment